# ⚡ VoltForge AI - Model Fine-Tuning on Google Colab

This notebook fine-tunes **Qwen 2.5 Coder (1.5B Instruct)** on the curated VoltForge Electronics Dataset (circuit safety, board pinouts, DRC rules, and MCU C++ firmware synthesis), then exports it directly into a standalone **Q4_K_M GGUF** file for `Voltforge_AI`.

### Prerequisites:
1. Make sure you are using a **T4 GPU** runtime: `Runtime` ➔ `Change runtime type` ➔ select `T4 GPU`.

In [ ]:
# Step 1: Check GPU availability
!nvidia-smi

In [ ]:
# Step 2: Install Unsloth & dependencies (PyTorch SDPA handles attention natively on T4)
%%capture
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo trl peft accelerate bitsandbytes

In [ ]:
# Step 3: Load Qwen 2.5 Coder 1.5B with 4-bit quantization
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # Auto detect (Float16 on T4)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("Model loaded with LoRA adapters ready for fine-tuning!")

### Step 4: Upload your dataset
Upload `voltforge_electronics_dataset.jsonl` from your computer using the cell below:

In [ ]:
# Upload voltforge_electronics_dataset.jsonl
from google.colab import files
import json
from datasets import Dataset

uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"Uploaded {filename}")

# Load records
records = []
with open(filename, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print(f"Loaded {len(records)} training records.")

# Format using Qwen Chat Template
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

raw_dataset = Dataset.from_list(records)
dataset = raw_dataset.map(formatting_prompts_func, batched=True)
print("Dataset formatted successfully!")

In [ ]:
# Step 5: Train the Model
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=200,  # ~10 minutes on T4 GPU
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print("Training completed successfully!")

In [ ]:
# Step 6: Export Directly to GGUF
print("Exporting fine-tuned model to Q4_K_M GGUF format...")
model.save_pretrained_gguf(
    "voltforge_ai_trained",
    tokenizer,
    quantization_method="q4_k_m"
)
print("Export complete!")

In [ ]:
# Step 7: Download the resulting GGUF file
from google.colab import files
import os

# Find exported gguf file
gguf_files = [f for f in os.listdir('voltforge_ai_trained') if f.endswith('.gguf')]
if gguf_files:
    target_file = os.path.join('voltforge_ai_trained', gguf_files[0])
    print(f"Downloading {target_file}...")
    files.download(target_file)
else:
    print("GGUF file not found in voltforge_ai_trained directory")